In [31]:
# Préparation des données
import numpy as np
import pandas as pd
import re
import random

# Vectorisation des ingrédients
from sklearn.feature_extraction.text import TfidfVectorizer

# Séparation des données
from sklearn.model_selection import train_test_split

# Modèles de Machine Learning
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

# Évaluation des modèles
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Modèles de régression importés avec succès !")

Modèles de régression importés avec succès !


In [7]:
data_final = pd.read_csv("../Dataset_ProjetIA_food/moroccan_recipes_cleaned.csv")

print("Dataset chargé avec succès !")
print("Dimensions :", data_final.shape)

data_final.head()

Dataset chargé avec succès !
Dimensions : (3291, 6)


,prompt,completion,Recipe_Name,Description,Instructions,Instruction_Length
0,Recipe Name : Duck and prune tagine\n Descript...,['Pour the oil into a heavy-based pan or casse...,Duck and prune tagine,Duck legs are the ultimate choice for low-and-...,['Pour the oil into a heavy-based pan or casse...,948
1,Recipe Name : Potato tagine with baked halloum...,['Heat the oven to 180°C fan/gas 6. Heat the o...,Potato tagine with baked halloumi,This North African-inspired potato dish is by ...,['Heat the oven to 180°C fan/gas 6. Heat the o...,737
2,Recipe Name : Merguez sausages with crispy chi...,['Heat the oven to 180°C fan/gas 6. Put the ch...,Merguez sausages with crispy chickpeas and pre...,"Merguez sausages with crispy, roasted chickpea...",['Heat the oven to 180°C fan/gas 6. Put the ch...,904
3,Recipe Name : Chicken tagine casserole\n Descr...,['Heat the oven to 180ºC fan/gas 6. Slash the ...,Chicken tagine casserole,Enjoy all the rich flavours of a chicken tagin...,['Heat the oven to 180ºC fan/gas 6. Slash the ...,486
4,Recipe Name : Preserved lemons\n Description :...,['Juice 4 of the lemons and set the juice asid...,Preserved lemons,With only two basic ingredients and some imagi...,['Juice 4 of the lemons and set the juice asid...,828


In [8]:
print("Colonnes disponibles :")
print(data_final.columns.tolist())

Colonnes disponibles :
['prompt', 'completion', 'Recipe_Name', 'Description', 'Instructions', 'Instruction_Length']


In [9]:
print(data_final.loc[0, 'prompt'])

Recipe Name : Duck and prune tagine
 Description : Duck legs are the ultimate choice for low-and-slow cooking, as they become fall-apart tender with ease. What’s more they’re flavoursome enough to stand up to lots of spices. It’s also a beautiful meat to match with fruit, which this tagine effortlessly proves. This recipe can be made ahead and frozen – see the tips below. Discover more brilliant batch cooking recipes.
 Ingredients : ['2 tbsp olive oil', '\xa02 cinnamon sticks', '3 red onions, finely sliced', '5 garlic cloves, finely sliced', '2 tbsp harissa', '4 tbsp ras el hanout', '2 tsp ground ginger', '4 large duck legs', '1 litre chicken stock', '2 x 400g tins chopped tomatoes', '2 preserved lemons', '400g tin chickpeas, drained', '150g pitted kalamata olives', '200g soft pitted prunes']
 Steps for preparation : ['Pour the oil into a heavy-based pan or casserole over a medium heat, then add the cinnamon sticks followed by the onions, garlic and a pinch of salt. Cook for about 5 mi

In [ ]:
print(data_final.loc[0, 'completion']a)

['Pour the oil into a heavy-based pan or casserole over a medium heat, then add the cinnamon sticks followed by the onions, garlic and a pinch of salt. Cook for about 5 minutes until the onions are just starting to soften, then add the harissa, ras el hanout and ground ginger. Stir for another minute.#', 'Add the duck legs, then cover with the stock and tinned tomatoes. Quarter the preserved lemons and scrape out the flesh, reserving half of it, then finely chop the peel and add the reserved flesh and peel to the pan. Leave to cook over a low heat, uncovered, for 2 hours.', 'Carefully remove the duck legs from the pan and use a fork to pull the meat from the bones – it should come away easily. Discard the bones and skin, then return the meat to the pan. Add the chickpeas, olives and prunes and cook for a further 1 hour.', 'Serve the tagine sprinkled with the chopped herbs, flaked almonds and pomegranate seeds, with rice on the side.']


In [ ]:

print("Nombre total de recettes :", len(data_final))

print("Ingrédients extraits :",
      data_final['Ingredients'].notna().sum())

print("Ingrédients manquants :",
      data_final['Ingredients'].isna().sum())


Nombre total de recettes : 3291
Ingrédients extraits : 3291
Ingrédients manquants : 0


In [16]:

import ast

def extract_ingredient_list(text):
    if pd.isna(text):
        return []

    try:
        return ast.literal_eval(text.strip())
    except (ValueError, SyntaxError):
        return []

data_final['Ingredients_List'] = data_final['Ingredients'].apply(
    extract_ingredient_list
)

data_final[['Recipe_Name', 'Ingredients_List']].head()


,Recipe_Name,Ingredients_List
0,Duck and prune tagine,"[2 tbsp olive oil, 2 cinnamon sticks, 3 red o..."
1,Potato tagine with baked halloumi,"[2 tbsp vegetable oil, 2 onions, sliced, 6 gar..."
2,Merguez sausages with crispy chickpeas and pre...,"[2 x 400g tins chickpeas, drained, 3 tbsp oliv..."
3,Chicken tagine casserole,"[8 small free-range chicken thighs, 2 medium r..."
4,Preserved lemons,[12 small unwaxed lemons (about 1kg; see tips)...


In [17]:

data_final['Number_of_Ingredients'] = data_final[
    'Ingredients_List'
].apply(len)

data_final[['Recipe_Name', 'Number_of_Ingredients']].head(10)


,Recipe_Name,Number_of_Ingredients
0,Duck and prune tagine,14
1,Potato tagine with baked halloumi,16
2,Merguez sausages with crispy chickpeas and pre...,13
3,Chicken tagine casserole,14
4,Preserved lemons,5
5,Whole baked chicken with North African spices,13
6,Root vegetable and lentil tagine,12
7,Fruity lamb and olive tagine,21
8,Spicy beef harira,13
9,"Lamb, harissa and feta burgers",14


In [18]:

def clean_ingredient(ingredient):
    ingredient = str(ingredient).lower().strip()
    ingredient = ingredient.replace('\xa0', ' ')
    ingredient = re.sub(r'\s+', ' ', ingredient)
    return ingredient

data_final['Ingredients_List'] = data_final['Ingredients_List'].apply(
    lambda ingredients: [
        clean_ingredient(item) for item in ingredients
        if clean_ingredient(item)
    ]
)

data_final[['Recipe_Name', 'Ingredients_List']].head()


,Recipe_Name,Ingredients_List
0,Duck and prune tagine,"[2 tbsp olive oil, 2 cinnamon sticks, 3 red on..."
1,Potato tagine with baked halloumi,"[2 tbsp vegetable oil, 2 onions, sliced, 6 gar..."
2,Merguez sausages with crispy chickpeas and pre...,"[2 x 400g tins chickpeas, drained, 3 tbsp oliv..."
3,Chicken tagine casserole,"[8 small free-range chicken thighs, 2 medium r..."
4,Preserved lemons,[12 small unwaxed lemons (about 1kg; see tips)...


In [19]:

data_final['Ingredients_Text'] = data_final['Ingredients_List'].apply(
    lambda ingredients: ' '.join(ingredients)
)

data_final[['Recipe_Name', 'Ingredients_Text']].head()


,Recipe_Name,Ingredients_Text
0,Duck and prune tagine,2 tbsp olive oil 2 cinnamon sticks 3 red onion...
1,Potato tagine with baked halloumi,"2 tbsp vegetable oil 2 onions, sliced 6 garlic..."
2,Merguez sausages with crispy chickpeas and pre...,"2 x 400g tins chickpeas, drained 3 tbsp olive ..."
3,Chicken tagine casserole,8 small free-range chicken thighs 2 medium red...
4,Preserved lemons,12 small unwaxed lemons (about 1kg; see tips) ...


In [20]:

print("Dimensions :", data_final.shape)
print("\nColonnes disponibles :")
print(data_final.columns.tolist())


Dimensions : (3291, 10)

Colonnes disponibles :
['prompt', 'completion', 'Recipe_Name', 'Description', 'Instructions', 'Instruction_Length', 'Ingredients', 'Ingredients_List', 'Number_of_Ingredients', 'Ingredients_Text']


In [22]:

data_final.to_csv(
    "../Dataset_ProjetIA_food/moroccan_recipes_model_ready.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dataset prêt pour le Machine Learning enregistré !")


Dataset prêt pour le Machine Learning enregistré !


In [24]:
print("Colonnes disponibles :")
print(data_final.columns.tolist())

Colonnes disponibles :
['prompt', 'completion', 'Recipe_Name', 'Description', 'Instructions', 'Instruction_Length', 'Ingredients', 'Ingredients_List', 'Number_of_Ingredients', 'Ingredients_Text']


In [26]:
data_final[['Recipe_Name', 'Ingredients_List', 'Number_of_Ingredients']].head()

,Recipe_Name,Ingredients_List,Number_of_Ingredients
0,Duck and prune tagine,"[2 tbsp olive oil, 2 cinnamon sticks, 3 red on...",14
1,Potato tagine with baked halloumi,"[2 tbsp vegetable oil, 2 onions, sliced, 6 gar...",16
2,Merguez sausages with crispy chickpeas and pre...,"[2 x 400g tins chickpeas, drained, 3 tbsp oliv...",13
3,Chicken tagine casserole,"[8 small free-range chicken thighs, 2 medium r...",14
4,Preserved lemons,[12 small unwaxed lemons (about 1kg; see tips)...,5


In [27]:
print(data_final[['Recipe_Name', 'Ingredients_List']].isnull().sum())

Recipe_Name         0
Ingredients_List    0
dtype: int64


In [28]:
print("Nombre total de recettes :", len(data_final))

Nombre total de recettes : 3291


In [32]:

random.seed(42)

training_data = []

# Parcourir chaque recette
for index, row in data_final.iterrows():

    ingredients = row['Ingredients_List']

    # Éviter les recettes sans ingrédients
    if not isinstance(ingredients, list) or len(ingredients) < 2:
        continue

    ingredients = list(set(ingredients))

    # Créer 10 paniers simulés par recette
    for i in range(10):

        # Choisir une partie des ingrédients de la recette
        proportion = random.uniform(0.3, 1.0)

        nombre = max(1, int(len(ingredients) * proportion))

        panier = random.sample(ingredients, nombre)

        # Ajouter éventuellement des ingrédients étrangers
        autres_ingredients = []

        for other_list in data_final['Ingredients_List']:
            if isinstance(other_list, list):
                autres_ingredients.extend(other_list)

        autres_ingredients = list(
            set(autres_ingredients) - set(ingredients)
        )

        if autres_ingredients:
            nombre_ajout = min(3, len(autres_ingredients))
            ajouts = random.sample(autres_ingredients, nombre_ajout)
            panier.extend(ajouts)

        # Calculer le score de correspondance
        score = len(set(panier) & set(ingredients)) / len(ingredients)

        training_data.append({
            'Recipe_Name': row['Recipe_Name'],
            'Panier': ' '.join(panier),
            'Ingredients_Recette': ' '.join(ingredients),
            'Score': score
        })

print("Exemples d'entraînement créés :", len(training_data))

Exemples d'entraînement créés : 26060


In [33]:
data_training = pd.DataFrame(training_data)

print("Dimensions :", data_training.shape)

data_training.head()

Dimensions : (26060, 4)


,Recipe_Name,Panier,Ingredients_Recette,Score
0,Duck and prune tagine,"4 large duck legs 400g tin chickpeas, drained ...",4 large duck legs 1 litre chicken stock 2 tbsp...,0.714286
1,Duck and prune tagine,1 litre chicken stock 4 tbsp ras el hanout 2 x...,4 large duck legs 1 litre chicken stock 2 tbsp...,0.285714
2,Duck and prune tagine,"3 red onions, finely sliced 400g tin chickpeas...",4 large duck legs 1 litre chicken stock 2 tbsp...,0.428571
3,Duck and prune tagine,2 x 400g tins chopped tomatoes 150g pitted kal...,4 large duck legs 1 litre chicken stock 2 tbsp...,0.857143
4,Duck and prune tagine,2 preserved lemons 2 x 400g tins chopped tomat...,4 large duck legs 1 litre chicken stock 2 tbsp...,0.500000


In [34]:
print("Score minimum :", data_training['Score'].min())
print("Score maximum :", data_training['Score'].max())

print("\nMoyenne des scores :")
print(data_training['Score'].mean())

Score minimum : 0.16666666666666666
Score maximum : 0.9761904761904762

Moyenne des scores :
0.6043338020257933


In [35]:
data_training[
    ['Recipe_Name', 'Panier', 'Ingredients_Recette', 'Score']
].head(10)

,Recipe_Name,Panier,Ingredients_Recette,Score
0,Duck and prune tagine,"4 large duck legs 400g tin chickpeas, drained ...",4 large duck legs 1 litre chicken stock 2 tbsp...,0.714286
1,Duck and prune tagine,1 litre chicken stock 4 tbsp ras el hanout 2 x...,4 large duck legs 1 litre chicken stock 2 tbsp...,0.285714
2,Duck and prune tagine,"3 red onions, finely sliced 400g tin chickpeas...",4 large duck legs 1 litre chicken stock 2 tbsp...,0.428571
3,Duck and prune tagine,2 x 400g tins chopped tomatoes 150g pitted kal...,4 large duck legs 1 litre chicken stock 2 tbsp...,0.857143
4,Duck and prune tagine,2 preserved lemons 2 x 400g tins chopped tomat...,4 large duck legs 1 litre chicken stock 2 tbsp...,0.500000
5,Duck and prune tagine,2 tbsp harissa 200g soft pitted prunes 150g pi...,4 large duck legs 1 litre chicken stock 2 tbsp...,0.857143
6,Duck and prune tagine,200g soft pitted prunes 2 tbsp olive oil 150g ...,4 large duck legs 1 litre chicken stock 2 tbsp...,0.714286
7,Duck and prune tagine,2 cinnamon sticks 2 preserved lemons 3 red oni...,4 large duck legs 1 litre chicken stock 2 tbsp...,0.357143
8,Duck and prune tagine,4 large duck legs 4 tbsp ras el hanout 150g pi...,4 large duck legs 1 litre chicken stock 2 tbsp...,0.785714
9,Duck and prune tagine,"3 red onions, finely sliced 2 tsp ground ginge...",4 large duck legs 1 litre chicken stock 2 tbsp...,0.857143


In [36]:
data_training['Texte_Complete'] = (
    "panier " + data_training['Panier'] +
    " recette " + data_training['Ingredients_Recette']
)

data_training[['Texte_Complete', 'Score']].head()

,Texte_Complete,Score
0,"panier 4 large duck legs 400g tin chickpeas, d...",0.714286
1,panier 1 litre chicken stock 4 tbsp ras el han...,0.285714
2,"panier 3 red onions, finely sliced 400g tin ch...",0.428571
3,panier 2 x 400g tins chopped tomatoes 150g pit...,0.857143
4,panier 2 preserved lemons 2 x 400g tins choppe...,0.500000


In [37]:
vectorizer = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(
    data_training['Texte_Complete']
)

y = data_training['Score']

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

Dimensions de X : (26060, 1000)
Dimensions de y : (26060,)


In [38]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_index, test_index = next(
    splitter.split(X, y, groups=data_training['Recipe_Name'])
)

X_train = X[train_index]
X_test = X[test_index]

y_train = y.iloc[train_index]
y_test = y.iloc[test_index]

print("Taille de X_train :", X_train.shape)
print("Taille de X_test :", X_test.shape)

print("Taille de y_train :", y_train.shape)
print("Taille de y_test :", y_test.shape)

Taille de X_train : (18240, 1000)
Taille de X_test : (7820, 1000)
Taille de y_train : (18240,)
Taille de y_test : (7820,)


In [39]:
print("Pourcentage entraînement :",
      round(len(train_index) / len(data_training) * 100, 2), "%")

print("Pourcentage test :",
      round(len(test_index) / len(data_training) * 100, 2), "%")

print("Recettes communes entre Train et Test :",
      len(set(data_training.iloc[train_index]['Recipe_Name']) &
          set(data_training.iloc[test_index]['Recipe_Name'])))

Pourcentage entraînement : 69.99 %
Pourcentage test : 30.01 %
Recettes communes entre Train et Test : 0


In [40]:
vectorizer = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 2)
)

X_train = vectorizer.fit_transform(
    data_training.iloc[train_index]['Texte_Complete']
)

X_test = vectorizer.transform(
    data_training.iloc[test_index]['Texte_Complete']
)

y_train = data_training.iloc[train_index]['Score']
y_test = data_training.iloc[test_index]['Score']

print("Vectorisation terminée !")
print("X_train :", X_train.shape)
print("X_test :", X_test.shape)

Vectorisation terminée !
X_train : (18240, 1000)
X_test : (7820, 1000)


In [41]:
models = {
    "Linear Regression": LinearRegression(),

    "KNN Regressor": KNeighborsRegressor(
        n_neighbors=5
    ),

    "Decision Tree Regressor": DecisionTreeRegressor(
        random_state=42
    ),

    "MLP Regressor": MLPRegressor(
        hidden_layer_sizes=(100, 50),
        activation='relu',
        solver='adam',
        max_iter=300,
        early_stopping=True,
        random_state=42
    )
}

print("Les 4 modèles sont initialisés !")

Les 4 modèles sont initialisés !


In [42]:
import time

trained_models = {}

for name, model in models.items():

    print(f"\nEntraînement : {name}")

    start_time = time.time()

    model.fit(X_train, y_train)

    end_time = time.time()

    trained_models[name] = model

    print("Entraînement terminé !")
    print("Temps :", round(end_time - start_time, 2), "secondes")

print("\nTous les modèles ont été entraînés !")


Entraînement : Linear Regression
Entraînement terminé !
Temps : 1.04 secondes

Entraînement : KNN Regressor
Entraînement terminé !
Temps : 0.01 secondes

Entraînement : Decision Tree Regressor
Entraînement terminé !
Temps : 24.87 secondes

Entraînement : MLP Regressor
Entraînement terminé !
Temps : 22.19 secondes

Tous les modèles ont été entraînés !


In [43]:
print("Modèles entraînés :")

for name in trained_models:
    print("-", name)

Modèles entraînés :
- Linear Regression
- KNN Regressor
- Decision Tree Regressor
- MLP Regressor


In [44]:
import joblib
import os

os.makedirs("../Models", exist_ok=True)

for name, model in trained_models.items():

    filename = name.lower().replace(" ", "_") + ".pkl"

    joblib.dump(
        model,
        f"../Models/{filename}"
    )

joblib.dump(vectorizer, "../Models/tfidf_vectorizer.pkl")

print("Les 4 modèles et le vectorizer ont été sauvegardés !")

Les 4 modèles et le vectorizer ont été sauvegardés !


In [45]:
data_training.to_csv(
    "../Dataset_ProjetIA_food/data_training.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Données d'entraînement sauvegardées !")

Données d'entraînement sauvegardées !
